<a href="https://colab.research.google.com/github/SolJosefinaMercado/ProcesamientoDelHabla/blob/main/TP2_Procesamiento_PDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP2 Procesamiento del Habla
### Sol Josefina Mercado DNI 38.442.698

1. Adquisición y metadatos
Descarga: Utilizando las librerías requests o urllib, descarga un archivo PDF desde un enlace público (por ejemplo, un artículo científico de arXiv o un reporte institucional).

Utilizaré: INDEC "Encuesta de ocupacion hotelera - Noviembre 2025"

Instalacion de las librerias necesarias
`pdfplumber` , `pymupdf` ,` pandas` y `requests`

In [1]:
!pip install pdfplumber pymupdf pandas requests -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 33.0 MB/s eta 0:00:00


Carga de librerias necesarias

In [2]:
# @title
import pdfplumber
import pymupdf          # PyMuPDF. Forma equivalente y muy usada: "import fitz"
import pandas as pd
import requests
import os

print("pdfplumber:", pdfplumber.__version__)
print("pymupdf:", pymupdf.__doc__.splitlines()[0] if pymupdf.__doc__ else pymupdf.version)
print("pandas:", pd.__version__)

pdfplumber: 0.11.10
pymupdf: PyMuPDF 1.28.2: Python bindings for the MuPDF 1.28.2 library.
pandas: 2.2.3



## 1 Adquisicion y metadatos

Descargo el informe de INDEC "Encuesta de ocupacion hotelera - Noviembre 2025"

Descargar el PDF y almacena en disco como  "ocupacion_hotelera_INDEC.pdf".


In [3]:
# @title
URL_PDF = "https://www.indec.gob.ar/uploads/informesdeprensa/eoh_01_265BAAB03CBF.pdf"
NOMBRE_ARCHIVO = "ocupacion_hotelera_INDEC.pdf"

respuesta = requests.get(URL_PDF)
# codigo de estado 2xx = exito en la descarga
print("Código de estado HTTP:", respuesta.status_code)

with open(NOMBRE_ARCHIVO, "wb") as f:
    f.write(respuesta.content)

tamano_kb = os.path.getsize(NOMBRE_ARCHIVO) / 1024
print(f"Archivo guardado como '{NOMBRE_ARCHIVO}' ({tamano_kb:.1f} KB)")


Código de estado HTTP: 200
Archivo guardado como 'ocupacion_hotelera_INDEC.pdf' (2057.5 KB)



## 2 Importar el PDF

Ahora que tenemos el archivo en disco, vamos a **abrirlo** con las dos librerías y comparar cómo se comporta cada una.

2.1 Con `pdfplumber`


In [4]:
pdf_plumber = pdfplumber.open(NOMBRE_ARCHIVO)

print("Tipo de objeto:", type(pdf_plumber))
print("Cantidad de páginas:", len(pdf_plumber.pages))

Tipo de objeto: <class 'pdfplumber.pdf.PDF'>
Cantidad de páginas: 21


2.2 Con `PyMuPDF`

In [5]:
# @title
doc = pymupdf.open(NOMBRE_ARCHIVO)

print("Tipo de objeto:", type(doc))
print("Cantidad de páginas:", doc.page_count)

Tipo de objeto: <class 'pymupdf.Document'>
Cantidad de páginas: 21


`notas`

`pdf plumber` representa cada pagina como un conjunto de objetos geometricos con coordenadas
* caracteres = page.chars
* lineas = page.lines
* rectangulos = page.rects
* imagenes = page.images
Puede reconstruir una grilla a partir de donde estan dibujadas las lineas y donde cae cada caracter

`PyMuPDF` es una interfaz de python sobre MuPDF, pensado para renderizar documentos, dibujarlos en pantalla.

* document es un objeto mas general: tiene paginas, metadatos, indice y enlaces. Y ademas permite editar, guardar y convertir. El texto lo entrega en distintos niveles de estructura ("text", "blocks", "dict", "rawdict")

`En resumen:`

* **PyMuPDF modela el PDF como un documento para mostrar y manipular, y pdfplumber lo modela como una colección de piezas geométricas para analizar.**

* Por eso en el notebook se usa PyMuPDF para los metadatos y el texto crudo, y pdfplumber para las tablas

## 3 Metadatos

Muestra por pantalla los metadatos del archivo:

La **metadata** de un PDF es información *sobre* el documento (no es el contenido en sí): autor, fecha de creación, título, software con el que fue generado, etc. Se guarda en una sección especial del archivo.

#### Metadata con `PyMuPDF` (contiene cuatro valores de matadatos extra, que pdfplumber no extrae)


### 3.1 Metadata con `PyMuPDF`

In [6]:
# @title
metadata_mupdf = doc.metadata
metadata_mupdf

{'format': 'PDF 1.7',
 'title': 'Encuesta de ocupación hotelera. Noviembre de 2025',
 'author': 'INDEC',
 'subject': 'Encuesta de ocupación hotelera',
 'keywords': '',
 'creator': 'Adobe InDesign 21.1 (Windows)',
 'producer': 'Adobe PDF Library 18.0',
 'creationDate': "D:20260119112801-03'00'",
 'modDate': "D:20260120141444-03'00'",
 'trapped': '',
 'encryption': None}

pymuPDF contiene los siguientes metadatos que NO contiene el dict de metadatos de pdf plumber:

* encryption
* keywords (sin valor)
* trapped (sin valor - `pdfplumber = False` )
* format (PDF 1.7),

## 'creationDate': "D:20260119112801-03'00'"

* D: indica que es una fecha
* 2026 Año
* 01 Mes
* 19 Día
* 11 Hora (24 horas)
* 28 Minutos
* 01 Segundos
* -03'00'Zona horaria



In [7]:
# @title
from datetime import datetime

# datetime.strptime(texto, formato) lee un texto y lo convierte en un objeto datetime, siguiendo un patrón que vos le indicás con códigos

valor = metadata_mupdf["creationDate"]
print('creation day', valor)

# valor limpio descarta los primeros dos caracteres y quita las comillas simples
valor_limpio = valor[2:].replace("'", "")
print('valor limpio', valor_limpio)

# si en fecha se quiere conservar la zona horaria %z
fecha = datetime.strptime(valor_limpio, "%Y%m%d%H%M%S%z").strftime("%d/%m/%Y %H:%M:%S")
print('Fecha formateada', fecha)

creation day D:20260119112801-03'00'
valor limpio 20260119112801-0300
Fecha formateada 19/01/2026 11:28:01


### 3.2 Modificacion de metadata

In [8]:
nueva_metadata = {
    "author": "Sol Josefina Mercado",
    "keywords": "EOH; ocupación hotelera; turismo; pernoctaciones; viajeros residentes y no residentes; tasa de ocupación; estadía promedio; regiones turísticas; Argentina; INDEC; noviembre 2025",
    "creator": "Taller de manipulación de PDFs con Python",
    'modDate': "D:20261007203600-03'00'" # 7/10/2026 - 20:36:00 hs
}

doc.set_metadata(nueva_metadata)
doc.save("ocupacion_hotelera_INDEC_modificado.pdf")

print("Nuevo archivo guardado: ocupacion_hotelera_INDEC_modificado.pdf")

Nuevo archivo guardado: ocupacion_hotelera_INDEC_modificado.pdf


In [9]:
# @title
doc_modificado = pymupdf.open("ocupacion_hotelera_INDEC_modificado.pdf")
doc_modificado.metadata

{'format': 'PDF 1.7',
 'title': 'Encuesta de ocupación hotelera. Noviembre de 2025',
 'author': 'Sol Josefina Mercado',
 'subject': 'Encuesta de ocupación hotelera',
 'keywords': 'EOH; ocupación hotelera; turismo; pernoctaciones; viajeros residentes y no residentes; tasa de ocupación; estadía promedio; regiones turísticas; Argentina; INDEC; noviembre 2025',
 'creator': 'Taller de manipulación de PDFs con Python',
 'producer': 'Adobe PDF Library 18.0',
 'creationDate': "D:20260119112801-03'00'",
 'modDate': "D:20261007203600-03'00'",
 'trapped': '',
 'encryption': None}

## 4 Análisis estructural del texto
### Extracción cruda:
Extrae el texto del PDF.

Identifica la función de la librería elegida que te ayuda en esta tarea y te permite ver el texto extraído lo más crudo posible.

Análisis de separadores: Observa y describe cómo están separados los párrafos, las secciones y los capítulos.

Identificación de caracteres: ¿Qué caracteres especiales funcionan como separadores de línea o de bloque en este archivo específico? (Ej: \n, \r, \t o secuencias de espacios).

In [39]:
# La función elegida es page.get_text("text") de PyMuPDF. Devuelve el texto en el orden en que está guardado, sin unir líneas, sin quitar guiones ni corregir nada. Para verlo realmente crudo, el notebook lo imprime con repr()

pagina = doc[17] #selecciono una pagina

# nivel 1 contenido real del pdf
crudo_pdf = pagina.read_contents()
print('Lo mas crudo que existe dentro del archivo: \n- No es texto, sino instrucciones de dibujo está codificadas')
print()
print(crudo_pdf[:50])


Lo mas crudo que existe dentro del archivo: 
- No es texto, sino instrucciones de dibujo está codificadas

b'q\n/GS0 gs\n/Fm0 Do\nQ\nBT\n/CS0 cs 0.45 0.448 0.466  s'


In [ ]:
# consulta Claude: con la libreria pymupdf puedo hacer una extraccion hasta en 3 niveles de crudeza del pdf, en el nivel 1 donde se observan las instrucciones de dibujo, estas son las instruciones de renderizado?

# Sí. Lo que ves en ese nivel es el content stream (flujo de contenido) de la página: la secuencia de operadores que un visor de PDF ejecuta, en orden, para "pintar" la página. Es correcto llamarlas instrucciones de renderizado o instrucciones de dibujo. El término técnico de la especificación PDF es operadores del flujo de contenido

'''
q                          ← guarda el estado gráfico
1 0 0 1 56.7 780.2 cm      ← transforma coordenadas
BT                         ← empieza un bloque de texto
/F1 8 Tf                   ← elige fuente F1, tamaño 8
56.7 30.1 Td               ← se mueve a una posición
(5/21) Tj                  ← dibuja ese texto
ET                         ← termina el bloque de texto
0.8 0.8 0.8 rg             ← color de relleno
56 700 480 0.5 re f        ← dibuja un rectángulo relleno (una línea)
/Im1 Do                    ← dibuja una imagen o un objeto externo
Q                          ← restaura el estado
'''

In [42]:
# nivel 2 caracter con su posicion y fuente

# Con la opción "dict", PyMuPDF devuelve un diccionario con toda la estructura de la página, organizada en niveles, como cajas dentro de cajas:

# extraigo con .get_text
crudo = pagina.get_text('dict', sort=True) # sort=True da el orden visual

# accedo por niveles e indices
primer_span = crudo['blocks'][0]['lines'][0]['spans'][0]

print('Primer span (diccionario completo con sus atributos):')
print(primer_span)
print()
print('Texto del primer span:')
print(primer_span['text'])

# cuantos spans tengo = Recordá que un span se corta solo cuando cambia el formato
spans = crudo['blocks'][0]['lines'][0]['spans']

print(len(spans))


Primer span (diccionario completo con sus atributos):
{'size': 14.0, 'flags': 4, 'bidi': 0, 'char_flags': 16, 'font': 'HelveticaNeueLTStd-Md', 'color': 5263185, 'alpha': 255, 'ascender': 0.9520000219345093, 'descender': -0.22100000083446503, 'text': 'Notas técnicas y definiciones utilizadas', 'origin': (70.91100311279297, 84.000244140625), 'bbox': (70.91100311279297, 70.6722412109375, 325.6269226074219, 87.09424591064453)}

Texto del primer span:
Notas técnicas y definiciones utilizadas
1


span  ← último nivel de la estructura

 ├── 'text'   → el contenido

 ├── 'font'   → la fuente

 ├── 'size'   → el tamaño

 ├── 'flags'  → estilo (negrita, cursiva…)

 ├── 'color'

 └── 'bbox'   → la posición (x0, y0, x1, y1)

{'spans': [{'size': 14.0, 'flags': 4, 'bidi': 0, 'char_flags': 16, 'font': 'HelveticaNeueLTStd-Md', 'color': 5263185, 'alpha': 255, 'ascender': 0.9520000219345093, 'descender': -0.22100000083446503,

'chars':

[
{'origin': (70.91100311279297, 84.000244140625), 'bbox': (70.91100311279297, 70.6722412109375, 81.01900482177734, 87.09424591064453), 'c': 'N', 'synthetic': False}


*Span: primera unidad minima de texto*

Por qué pasa: PyMuPDF devuelve los bloques en el orden en que están escritos dentro del archivo, no en el orden en que se ven en la página.

Con `sort=True` se obtiene el orden de lectura visual.

In [12]:
# nivel 3, texto plano
# pagina 18
texto = pagina.get_text('text')[:1000]
print(texto)

18/21
Notas técnicas y definiciones utilizadas
La EOH se inició en 2004 en 17 destinos turísticos del país. En 2005, la muestra se amplió a 39, con el 
objetivo de lograr representatividad para las diferentes regiones turísticas del país. Los destinos fueron 
seleccionados entre aquellos que reunían 80% de las plazas disponibles en cada región. En 2006, el 
operativo se amplió a 44 destinos turísticos y, en 2009, se agregaron 4 más. A partir de enero de 2025, 
se relevan 51 destinos en el país, distribuidos en siete regiones turísticas. La Ciudad Autónoma de Bue-
nos Aires (CABA) es considerada como una región turística debido a su importancia y características 
particulares. 
Se cuenta con un registro hotelero que contiene la nómina de todos los establecimientos hoteleros 
y parahoteleros con sus principales características: categoría, número de habitaciones o unidades 
disponibles, número de plazas y equipamiento informático. Se actualiza mediante la incorporación 
de los nuevos esta

Identificación de caracteres: ¿Qué caracteres especiales funcionan como separadores de línea o de bloque en este archivo específico? (Ej: \n, \r, \t o secuencias de espacios).

Texto extraido:

'18/21 (n_página) `\n` Notas técnicas y definiciones utilizadas (título) `\n`La EOH se inició en 2004 en 17 destinos turísticos del país. En 2005, la muestra se amplió a 39, con el `\n`objetivo de lograr representatividad para las diferentes regiones turísticas del país. Los destinos fueron `\n`seleccionados entre aquellos que reunían

Cambio de parrafo `. \nSe cuenta`

## Extraccion del texto completo PyMuPDF

In [13]:
# @title
texto_completo_mupdf = ""

for pag in doc:
  texto_pagina = pagina.get_text()
  if texto_pagina:
    texto_completo_mupdf += texto_pagina + "\n"

print('cantidad de caracteres extraidos pymupdf:', len(texto_completo_mupdf))
print('cantidad de palabras aproximadas', len(texto_completo_mupdf.split()))
print('-' * 100)
print(texto_completo_mupdf[:1000])

cantidad de caracteres extraidos pymupdf: 77343
cantidad de palabras aproximadas 11487
----------------------------------------------------------------------------------------------------
18/21
Notas técnicas y definiciones utilizadas
La EOH se inició en 2004 en 17 destinos turísticos del país. En 2005, la muestra se amplió a 39, con el 
objetivo de lograr representatividad para las diferentes regiones turísticas del país. Los destinos fueron 
seleccionados entre aquellos que reunían 80% de las plazas disponibles en cada región. En 2006, el 
operativo se amplió a 44 destinos turísticos y, en 2009, se agregaron 4 más. A partir de enero de 2025, 
se relevan 51 destinos en el país, distribuidos en siete regiones turísticas. La Ciudad Autónoma de Bue-
nos Aires (CABA) es considerada como una región turística debido a su importancia y características 
particulares. 
Se cuenta con un registro hotelero que contiene la nómina de todos los establecimientos hoteleros 
y parahoteleros con sus pri

# Análisis de frecuencias (NLP básico)

In [45]:
# @title
# búsqueda de texto utilizando PyMuPDF
# ventajas de pymu sobre plumber (sobre texto)
# velocidad, mayor cantidad de fromatos dict-word-blocks
# un solo objeto para todo el contenido "doc"

import pymupdf

ARCHIVO = "ocupacion_hotelera_INDEC.pdf"
with pymupdf.open(ARCHIVO) as doc:
    for i, p in enumerate(doc):
        texto_pagina = p.get_text()
        if texto_pagina.strip():
            print(f"Página {i+1}: {len(texto_pagina)} caracteres")

Página 1: 121 caracteres
Página 2: 6658 caracteres
Página 3: 1286 caracteres
Página 4: 2870 caracteres
Página 5: 3402 caracteres
Página 6: 1705 caracteres
Página 7: 2369 caracteres
Página 8: 2007 caracteres
Página 9: 2029 caracteres
Página 10: 1947 caracteres
Página 11: 617 caracteres
Página 12: 1725 caracteres
Página 13: 2676 caracteres
Página 14: 2705 caracteres
Página 15: 2937 caracteres
Página 16: 4233 caracteres
Página 17: 4146 caracteres
Página 18: 3682 caracteres
Página 19: 1376 caracteres
Página 20: 3686 caracteres
Página 21: 3277 caracteres


### Selecciono la Página 18: 3682 caracteres

In [60]:
# @title
# Volvemos a abrir el documento ya que el bloque 'with' anterior lo cerró
doc = pymupdf.open(ARCHIVO)
pagina = doc[17] # pagina 18
texto_pagina = pagina.get_text()
print(texto_pagina[:1000])

18/21
Notas técnicas y definiciones utilizadas
La EOH se inició en 2004 en 17 destinos turísticos del país. En 2005, la muestra se amplió a 39, con el 
objetivo de lograr representatividad para las diferentes regiones turísticas del país. Los destinos fueron 
seleccionados entre aquellos que reunían 80% de las plazas disponibles en cada región. En 2006, el 
operativo se amplió a 44 destinos turísticos y, en 2009, se agregaron 4 más. A partir de enero de 2025, 
se relevan 51 destinos en el país, distribuidos en siete regiones turísticas. La Ciudad Autónoma de Bue-
nos Aires (CABA) es considerada como una región turística debido a su importancia y características 
particulares. 
Se cuenta con un registro hotelero que contiene la nómina de todos los establecimientos hoteleros 
y parahoteleros con sus principales características: categoría, número de habitaciones o unidades 
disponibles, número de plazas y equipamiento informático. Se actualiza mediante la incorporación 
de los nuevos esta

### Tokenización: Extrae el texto y conviértelo a minúsculas.

In [57]:
texto_minuscula = texto_pagina.lower()
print(texto_minuscula[:100])

18/21
notas técnicas y definiciones utilizadas
la eoh se inició en 2004 en 17 destinos turísticos de


In [65]:
# @title
import re

# Une palabras cortadas con guion al final de línea: "bue-\nnos" -> "buenos"
texto_unido = re.sub(r"(\w)-\n(\w)", r"\1\2", texto_minuscula)

# Recién ahora reemplazo los saltos de línea por espacios
texto_sin_saltos = texto_unido.replace("\n", " ").replace("\r", " ").replace("/", " ")

print('texto sin saltos y en un solo string')
print()
print(texto_sin_saltos[:100])

texto sin saltos y en un solo string

18 21 notas técnicas y definiciones utilizadas la eoh se inició en 2004 en 17 destinos turísticos de


### Limpieza: Elimina las stop words

In [77]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download('punkt_tab') # Added to download the missing resource
nltk.download('stopwords')


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [78]:
tokens = word_tokenize(texto_sin_saltos, language="spanish")
print("Cantidad de tokens:", len(tokens))
print(tokens[:30])

Cantidad de tokens: 632
['18', '21', 'notas', 'técnicas', 'y', 'definiciones', 'utilizadas', 'la', 'eoh', 'se', 'inició', 'en', '2004', 'en', '17', 'destinos', 'turísticos', 'del', 'país', '.', 'en', '2005', ',', 'la', 'muestra', 'se', 'amplió', 'a', '39', ',']


In [79]:
stop_es = set(stopwords.words("spanish"))

# isalpha() elimina la puntuación y también los números, quedándonos solo con palabras limpias
tokens_limpios = [t for t in tokens if t.isalpha() and t not in stop_es]

print("Tokens después de la limpieza:", len(tokens_limpios))
print(tokens_limpios[:30])

Tokens después de la limpieza: 289
['notas', 'técnicas', 'definiciones', 'utilizadas', 'eoh', 'inició', 'destinos', 'turísticos', 'país', 'muestra', 'amplió', 'objetivo', 'lograr', 'representatividad', 'diferentes', 'regiones', 'turísticas', 'país', 'destinos', 'seleccionados', 'aquellos', 'reunían', 'plazas', 'disponibles', 'cada', 'región', 'operativo', 'amplió', 'destinos', 'turísticos']


In [80]:
# FreqDist no es una librería sino una clase de NLTK (Natural Language Toolkit), la librería de procesamiento de lenguaje natural
# Está en el módulo nltk.probability
# Frequency Distribution (distribución de frecuencias)
# FreqDist funciona como un diccionario donde cada clave es una palabra y cada valor es la cantidad de veces que aparece
# Recorre la lista token por token y va sumando: si la palabra no existe, la agrega con valor 1, y si ya existe, le suma 1
from nltk.probability import FreqDist

frecuencia_palabras = FreqDist(tokens_limpios)

print("Las 15 palabras más frecuentes son:")
for palabra, frecuencia in frecuencia_palabras.most_common(15):
    print(f"{palabra}: {frecuencia}")

Las 15 palabras más frecuentes son:
habitaciones: 11
unidades: 10
mes: 10
plazas: 9
disponibles: 8
establecimientos: 8
referencia: 8
establecimiento: 6
total: 6
destinos: 4
cada: 4
número: 4
ocupadas: 4
país: 3
hoteleros: 3


In [81]:
top15 = frecuencia_palabras.most_common(15)
df_frecuencias = pd.DataFrame(top15, columns=["palabra", "frecuencia"])

In [83]:
import plotly.express as px

fig = px.bar(df_frecuencias, x="palabra", y="frecuencia", title="Frecuencia de palabras en el texto",
             template = 'plotly_dark')
fig.show()

Stop words de dominio

In [108]:
from nltk.corpus import stopwords

stop_es = set(stopwords.words("spanish"))
stop_contexto = {"fuente", "indec", "dirección", "estadísticas", "básicas","destinos",
                "balanza", "pagos", "cuadro", "gráfico", "encuesta", "cada", "establecimiento", "unidad", "pais", "total", "eoh", "turísticos"}

stop_total = stop_es | stop_contexto

tokens_limpios_dom = [t for t in tokens if t not in stop_total and t.isalpha()]

frecuencia_palabras_dom = FreqDist(tokens_limpios_dom)

print("Las 15 palabras más frecuentes son:")
for palabra, frecuencia in frecuencia_palabras_dom.most_common(15):
    print(f"{palabra}: {frecuencia}")



Las 15 palabras más frecuentes son:
habitaciones: 11
unidades: 10
mes: 10
plazas: 9
disponibles: 8
establecimientos: 8
referencia: 8
número: 4
ocupadas: 4
país: 3
hoteleros: 3
parahoteleros: 3
cantidad: 3
definiciones: 2
muestra: 2


In [109]:
top15_dom = frecuencia_palabras_dom.most_common(15)
df_frecuencias_dom = pd.DataFrame(top15_dom, columns=["palabra", "frecuencia"])
fig = px.bar(df_frecuencias, x="palabra", y="frecuencia", title="Frecuencia de palabras en el texto",
             template = 'plotly_dark')
fig.show()

---

## Extracción de datos tabulares
Busca una página en el PDF que contenga una tabla de datos.

Extrae dicha tabla y conviértela en un DataFrame de Pandas.

Realiza una limpieza básica de la tabla (nombrar columnas, eliminar filas vacías).

Sugerencia: Se recomienda el uso de pdfplumber o tabula-py.

In [14]:
# busqueda de tablas utilizando pdf plumber

ARCHIVO = "ocupacion_hotelera_INDEC.pdf"
with pdfplumber.open(ARCHIVO) as pdf:
    for i, p in enumerate(pdf.pages):
        n = len(p.find_tables())
        if n > 0:
            print(f"Página {i+1}: {n} tabla(s)")

Página 3: 7 tabla(s)
Página 5: 1 tabla(s)
Página 6: 4 tabla(s)
Página 7: 11 tabla(s)
Página 8: 6 tabla(s)
Página 9: 1 tabla(s)
Página 10: 1 tabla(s)
Página 12: 1 tabla(s)
Página 13: 2 tabla(s)
Página 14: 2 tabla(s)
Página 15: 3 tabla(s)
Página 16: 1 tabla(s)
Página 17: 1 tabla(s)
Página 20: 1 tabla(s)


In [15]:
pagina_tabla = pdf_plumber.pages[16] # pagina 17, cuadro 14
tabla_raw = pagina_tabla.extract_table()

In [16]:
print("Tipo de dato:", type(tabla_raw))
print("Cantidad de filas:", len(tabla_raw))
print("Primeras 3 filas:")
for fila in tabla_raw[:3]:
    print(fila)

Tipo de dato: <class 'list'>
Cantidad de filas: 60
Primeras 3 filas:
['Destinos turísticos', 'Habitaciones y unidades*', None, None, None, None, None, None, None, 'Plazas*', None, None, None, None, None, None, None]
[None, 'Disponibles', None, 'Ocupadas', None, 'Tasa de\nocupación', None, 'Tasa de\nocupación\nfines de\nsemana', None, 'Disponibles', None, 'Ocupadas', None, 'Tasa de\nocupación', None, 'Tasa de\nocupación\nfines de\nsemana', None]
['', 'Habitaciones', None, None, None, '%', None, None, None, 'Plazas', None, None, None, '%', None, None, None]


In [17]:
encabezado = tabla_raw[:2]
print(encabezado)

[['Destinos turísticos', 'Habitaciones y unidades*', None, None, None, None, None, None, None, 'Plazas*', None, None, None, None, None, None, None], [None, 'Disponibles', None, 'Ocupadas', None, 'Tasa de\nocupación', None, 'Tasa de\nocupación\nfines de\nsemana', None, 'Disponibles', None, 'Ocupadas', None, 'Tasa de\nocupación', None, 'Tasa de\nocupación\nfines de\nsemana', None]]


In [18]:
# El encabezado esta compuesto por dos filas, ya que son dos categorias con subcategorias
# unifico esas dos filas para poder tener un unico encabezado

fila_grupo, fila_sub = tabla_raw[0], tabla_raw[1] #primer y segunda fila

nombres = []
#grupo_actual es para que la variable exista antes de empezar el bucle. En cada vuelta, el código la usa para armar el nombre de la columna, pero solo le asigna un valor cuando encuentra un grupo nuevo en la primera fila
grupo_actual = ""
# con zip recorro las listas en paralelo y de a pares
# grupo es el contenido de tabla_raw[0], sub tabla_raw[1]
for grupo, sub in zip(fila_grupo, fila_sub):
    if grupo:
        # si existe contenido en la primer fila elimina los astericos y los espacios al inicio y final
        grupo_actual = grupo.replace("*", "").strip()
    if sub:
        # si existe contenido en la segunda fila, suma a la lista nombres el nombre del grupo + nombre de sub grupo
        # chr(10) es el carácter \n, reemplaza por espacio
        nombres.append(f"{grupo_actual} - {sub.replace(chr(10), ' ')}")
    else:
        # expresion condicional A if condicion, else B
        # Si ya hay nombres, es una columna de nota. nombres[-1] es el último elemento de la lista, o sea, el nombre de la columna de datos que está justo a su izquierda. Se le agrega(nota)
        nombres.append(f"{nombres[-1]} (nota)" if nombres else "destino")

nombres[0] = "destino"
print(nombres)

['destino', 'Habitaciones y unidades - Disponibles', 'Habitaciones y unidades - Disponibles (nota)', 'Habitaciones y unidades - Ocupadas', 'Habitaciones y unidades - Ocupadas (nota)', 'Habitaciones y unidades - Tasa de ocupación', 'Habitaciones y unidades - Tasa de ocupación (nota)', 'Habitaciones y unidades - Tasa de ocupación fines de semana', 'Habitaciones y unidades - Tasa de ocupación fines de semana (nota)', 'Plazas - Disponibles', 'Plazas - Disponibles (nota)', 'Plazas - Ocupadas', 'Plazas - Ocupadas (nota)', 'Plazas - Tasa de ocupación', 'Plazas - Tasa de ocupación (nota)', 'Plazas - Tasa de ocupación fines de semana', 'Plazas - Tasa de ocupación fines de semana (nota)']


In [19]:
# crea el df con las filas de
df = pd.DataFrame(tabla_raw[3:], columns=nombres)
df = df.drop(columns=[c for c in df.columns if c.endswith("(nota)")])

In [20]:
df.head(10)

,destino,Habitaciones y unidades - Disponibles,Habitaciones y unidades - Ocupadas,Habitaciones y unidades - Tasa de ocupación,Habitaciones y unidades - Tasa de ocupación fines de semana,Plazas - Disponibles,Plazas - Ocupadas,Plazas - Tasa de ocupación,Plazas - Tasa de ocupación fines de semana
0,Región Buenos Aires,,,,,,,,
1,Bahía Blanca,17.370,10.549,"60,7","55,9",44.730,20.022,"44,8","44,6"
2,Mar del Plata,380.081,146.365,"38,5","53,3",1.020.898,343.265,"33,6","46,7"
3,Pinamar (2),...,...,...,...,...,...,...,...
4,Tandil,33.060,15.596,"47,2","60,1",112.800,32.756,"29,0","38,1"
5,Villa Gesell,99.383,24.497,"24,6","33,4",302.837,63.403,"20,9","27,0"
6,Región Córdoba,,,,,,,,
7,Ciudad de Córdoba,82.860,48.436,"58,5","59,3",190.080,87.635,"46,1","50,7"
8,La Falda,46.591,6.230,"13,4","18,6",136.268,13.319,"9,8","15,2"
9,Miramar,11.430,3.239,"28,3","45,7",32.628,7.341,"22,5","38,4"


# Transformo la tabla jerarquica original en formato tabular plano

In [21]:
# Reemplazo las celdas vacias o que solo tienen espacios por NaN para detectar las filas de regiones
import numpy as np

df = df.replace(r"^\s*$", np.nan, regex=True)

In [22]:
# marco las filas que pertenecen a informacion de Regiones
es_region = df['destino'].str.startswith('Región', na=False)

In [23]:
# Crea la columna "region": copia el nombre solo en esas filas y lo arrastra hacia abajo con ffill() hasta la próxima región
df['region'] = df['destino'].where(es_region).astype('string').ffill()

In [24]:
# Elimino la palabra Region de cada fila
df['region'] = df['region'].str.replace('Región', '', regex=False)

In [25]:
# df[~es_region] es un filtro booleano: se queda solo con las filas donde la condición es True, o sea, los destinos
# al borrar filas, el índice queda con huecos. Esto lo renumera desde 0 sin saltos
df = df[~es_region].reset_index(drop=True)

In [26]:
# ubico a region como primer columna
df = df[["region"] + [c for c in df.columns if c != "region"]]

In [27]:
# las filas con (2) al lado del nombre no contiene datos porque no existieron suficientes datos.
# elimino las filas

# listo todas las columnas menos region y destino
columnas_datos = [c for c in df.columns if c not in ("region", "destino")]

In [28]:
# para las col almacenadas, si contiene ... reemplaza por NaN
df[columnas_datos] = df[columnas_datos].replace("...", np.nan)

In [29]:
# hago el drop Na y reseteo index
df = df.dropna(subset=columnas_datos, how="all").reset_index(drop=True)

In [30]:
# guardo el df extraido
df.to_csv("ocupacion_hotelera_INDEC_limpio.csv", index=False)

In [31]:
df.head()

,region,destino,Habitaciones y unidades - Disponibles,Habitaciones y unidades - Ocupadas,Habitaciones y unidades - Tasa de ocupación,Habitaciones y unidades - Tasa de ocupación fines de semana,Plazas - Disponibles,Plazas - Ocupadas,Plazas - Tasa de ocupación,Plazas - Tasa de ocupación fines de semana
0,Buenos Aires,Bahía Blanca,17.370,10.549,"60,7","55,9",44.730,20.022,"44,8","44,6"
1,Buenos Aires,Mar del Plata,380.081,146.365,"38,5","53,3",1.020.898,343.265,"33,6","46,7"
2,Buenos Aires,Tandil,33.060,15.596,"47,2","60,1",112.800,32.756,"29,0","38,1"
3,Buenos Aires,Villa Gesell,99.383,24.497,"24,6","33,4",302.837,63.403,"20,9","27,0"
4,Córdoba,Ciudad de Córdoba,82.860,48.436,"58,5","59,3",190.080,87.635,"46,1","50,7"
